# 🧠 Advanced Data Augmentation with Keras

> **Module:** 03 — Deep Learning with Keras and TensorFlow  
> **Topic:** Data augmentation, normalisation, custom preprocessing  
> **Dataset:** CIFAR-10 (60,000 colour images, 10 classes)

---

## 📋 Overview

In this notebook I implement and experiment with **data augmentation** — a family of techniques that artificially expand a training dataset by applying controlled transformations to existing images. The goal is to make a model more robust by exposing it to variations it might see at inference time.

**What I build:**

| Part | Technique | Tool |
|---|---|---|
| Part 1 | Basic geometric augmentation | `ImageDataGenerator` |
| Part 2 | Feature-wise & sample-wise normalisation | `ImageDataGenerator.fit()` |
| Part 3 | Custom augmentation function (random noise) | `preprocessing_function=` |
| Part 4 | Visualise augmented images | `matplotlib` |
| Exercises | Apply all three techniques to a real image batch | Solved below |

**Learning goals:**
- Implement geometric augmentations (rotation, shift, shear, zoom, flip)
- Understand the difference between feature-wise and sample-wise normalisation
- Write a custom `preprocessing_function` that adds Gaussian noise
- Visualise the effect of each augmentation type on real images

## 🧩 Theory

### Why augment?

A model trained on a small fixed dataset memorises the exact pixel patterns it saw — it overfits. Data augmentation addresses this by generating **new training samples on-the-fly** from the existing ones, so the model never sees the same exact image twice.

**Telecom / RF analogy 📡:** Think of a channel equalizer trained on a limited set of channel snapshots. If you only train on calm, stationary channels, the equalizer collapses when it encounters multipath fading or Doppler shifts. Data augmentation is like synthetically generating diverse channel conditions during training — rotation ≈ phase rotation, zoom ≈ delay spread variation, noise injection ≈ AWGN — so the equalizer generalises to real-world conditions.

### Geometric augmentations

Each transformation applies a spatial mapping $T: \mathbb{R}^{H \times W} \rightarrow \mathbb{R}^{H \times W}$:

| Augmentation | Operation | Formula |
|---|---|---|
| Rotation by angle $\theta$ | Rotate pixel grid | $\begin{pmatrix}x'\\\\y'\end{pmatrix} = \begin{pmatrix}\cos\theta & -\sin\theta \\\\ \sin\theta & \cos\theta\end{pmatrix}\begin{pmatrix}x\\\\y\end{pmatrix}$ |
| Width / height shift by fraction $s$ | Translate | $x' = x + s \cdot W$ |
| Shear by angle $\phi$ | Shear mapping | $x' = x + y\tan\phi$ |
| Zoom by factor $z$ | Scale | $x' = x/z,\ y' = y/z$ |
| Horizontal flip | Mirror | $x' = W - 1 - x$ |

### Normalisation

**Feature-wise normalisation** computes statistics across the *entire dataset* and applies them to each sample:

$$x'_{i} = \frac{x_i - \mu_{\text{dataset}}}{\sigma_{\text{dataset}}}$$

**Sample-wise normalisation** computes statistics per *individual sample*:

$$x'_{i} = \frac{x_i - \mu_{\text{sample}}}{\sigma_{\text{sample}}}$$

Feature-wise is like calibrating an RF receiver against a known noise floor measured across many frames. Sample-wise is like AGC (Automatic Gain Control) — normalising each received burst independently regardless of the global signal environment.

### Gaussian noise injection

Adding noise $\epsilon \sim \mathcal{N}(0, \sigma^2)$ to each pixel:

$$x'_i = x_i + \epsilon_i, \quad \epsilon_i \sim \mathcal{N}(0, \sigma^2)$$

This is directly analogous to AWGN (Additive White Gaussian Noise) in communications — the model must learn to recognise objects even through channel noise, improving robustness.

## ⚙️ Part 0 — Setup & Dataset

I start by installing the required libraries, importing everything, and loading CIFAR-10. I normalise pixel values to $[0, 1]$ by dividing by 255 — this is standard preprocessing before augmentation so all operations happen in a consistent numerical range.

In [ ]:
# Install required libraries
!pip install tensorflow==2.16.2 matplotlib==3.9.1 scipy --quiet

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f"TensorFlow version: {tf.__version__}")

# Load CIFAR-10 — 50,000 training images, 10,000 test images
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalise pixel values from [0, 255] to [0.0, 1.0]
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

print(f"Training set:  {x_train.shape}  ({x_train.dtype})")
print(f"Test set:      {x_test.shape}   ({x_test.dtype})")
print(f"Pixel range:   [{x_train.min():.2f}, {x_train.max():.2f}]")

I display a 4×4 grid of raw CIFAR-10 training images to get a feel for the data before any augmentation is applied.

In [ ]:
CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

plt.figure(figsize=(10, 10))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(x_train[i])
    plt.title(CLASS_NAMES[y_train[i][0]], fontsize=8)
    plt.axis('off')
plt.suptitle('📥 CIFAR-10 — Raw Training Images', fontsize=14)
plt.tight_layout()
plt.show()

### 🖼️ Creating a Synthetic Sample Image

For the augmentation demonstrations I also create a simple synthetic image using PIL — a white background with a red square. This makes the geometric transformations visually obvious.

In [ ]:
from PIL import Image, ImageDraw

image = Image.new('RGB', (224, 224), color=(255, 255, 255))
draw  = ImageDraw.Draw(image)
draw.rectangle([(50, 50), (174, 174)], fill=(255, 0, 0))
image.save('sample.jpg')

plt.imshow(image)
plt.title('🖼️ Synthetic sample image (red square on white)')
plt.axis('off')
plt.show()
print("sample.jpg saved.")

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array

img_path = 'sample.jpg'
img = load_img(img_path)
x   = img_to_array(img)
x   = np.expand_dims(x, axis=0)

print(f"Image tensor shape: {x.shape}  (batch=1, H=224, W=224, C=3)")

## 🔄 Part 1 — Basic Geometric Augmentation

`ImageDataGenerator` applies stochastic geometric transformations to images at training time.

| Parameter | Value | Effect |
|---|---|---|
| `rotation_range` | 40° | Rotate by random angle in $[-40°, +40°]$ |
| `width_shift_range` | 0.2 | Shift left/right by up to $20\%$ of width |
| `height_shift_range` | 0.2 | Shift up/down by up to $20\%$ of height |
| `shear_range` | 0.2 | Shear up to $0.2$ radians |
| `zoom_range` | 0.2 | Zoom factor in $[0.8, 1.2]$ |
| `horizontal_flip` | True | Random mirror |
| `fill_mode` | 'nearest' | Fill new pixels with nearest neighbour |

In [ ]:
datagen_basic = ImageDataGenerator(
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, batch in enumerate(datagen_basic.flow(x, batch_size=1)):
    axes[i].imshow(batch[0].astype('uint8'))
    axes[i].set_title(f'🔄 Augmented {i+1}')
    axes[i].axis('off')
    if i >= 3:
        break
plt.suptitle('Part 1 — Basic Geometric Augmentations', fontsize=12)
plt.tight_layout()
plt.show()

Each call to `datagen_basic.flow()` samples fresh random parameters. The model seeing these variations learns that the object concept is invariant to spatial transformations.

## 📐 Part 2 — Feature-wise & Sample-wise Normalisation

**Feature-wise (global):** compute $\mu$ and $\sigma$ across the full dataset:
$$x'_{c,h,w} = \frac{x_{c,h,w} - \mu_c}{\sigma_c}$$
Requires `datagen.fit(data)` first.

**Sample-wise (local):** compute per individual image:
$$x'_{i} = \frac{x_i - \mu_{\text{sample}}}{\sigma_{\text{sample}}}$$

Feature-wise ≈ RF receiver calibrated against a global noise floor. Sample-wise ≈ AGC — normalise each burst independently.

In [ ]:
datagen_norm = ImageDataGenerator(
    featurewise_center=True,
    featurewise_std_normalization=True,
    samplewise_center=True,
    samplewise_std_normalization=True
)
datagen_norm.fit(x)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, batch in enumerate(datagen_norm.flow(x, batch_size=1)):
    display_img = np.clip(batch[0], 0, 255).astype('uint8')
    axes[i].imshow(display_img)
    axes[i].set_title(f'📐 Normalised {i+1}')
    axes[i].axis('off')
    if i >= 3:
        break
plt.suptitle('Part 2 — Feature-wise & Sample-wise Normalisation', fontsize=12)
plt.tight_layout()
plt.show()

> **Note:** After normalisation, pixel values are centred around 0 — they may be negative. Clipping to $[0, 255]$ for display is correct; the network sees the normalised values.

## ⚡ Part 3 — Custom Augmentation: Gaussian Noise

I define a custom `preprocessing_function` that injects AWGN into each image:
$$x'_i = x_i + \epsilon_i, \quad \epsilon_i \sim \mathcal{N}(0, \sigma^2), \quad \sigma = 0.1$$

Training on noisy images forces the network to learn denoising-aware, robust features — directly analogous to training a receiver under varying SNR conditions.

In [ ]:
def add_random_noise(image):
    """Add AWGN: x' = x + N(0, 0.1²)"""
    noise = np.random.normal(loc=0.0, scale=0.1, size=image.shape)
    return image + noise

datagen_noise = ImageDataGenerator(preprocessing_function=add_random_noise)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, batch in enumerate(datagen_noise.flow(x, batch_size=1)):
    display_img = np.clip(batch[0], 0, 255).astype('uint8')
    axes[i].imshow(display_img)
    axes[i].set_title(f'⚡ Noisy {i+1}')
    axes[i].axis('off')
    if i >= 3:
        break
plt.suptitle('Part 3 — Custom Augmentation: Gaussian Noise (σ=0.1)', fontsize=12)
plt.tight_layout()
plt.show()

## 📊 Part 4 — Visualise Augmented Variants

A 2×2 grid showing 4 different stochastic draws from the noise generator — demonstrating that each call produces a unique transformation of the original image.

In [ ]:
plt.figure(figsize=(8, 8))
for i, batch in enumerate(datagen_noise.flow(x, batch_size=1)):
    if i >= 4:
        break
    plt.subplot(2, 2, i + 1)
    plt.imshow(np.clip(batch[0], 0, 255).astype('uint8'))
    plt.title(f'🧪 Version {i+1}')
    plt.axis('off')
plt.suptitle('📊 Visualising Augmented Image Variants (Gaussian Noise)', fontsize=13)
plt.tight_layout()
plt.show()

---

## 🔢 Exercises — Solved

### Exercise 1 — Apply & Visualise Geometric Augmentations on Real Images

In [ ]:
!wget -q https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/RgP3JFNtPTZA34UmG3KZaA/sample-images.zip
!unzip -q sample-images.zip

In [ ]:
from tensorflow.keras.preprocessing.image import array_to_img

datagen_ex1 = ImageDataGenerator(
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

image_paths = [
    'sample_images/training_images1.jpg',
    'sample_images/training_images2.jpg',
    'sample_images/training_images3.jpg'
]

training_images = []
for image_path in image_paths:
    img = load_img(image_path, target_size=(224, 224))
    img_array = img_to_array(img)
    training_images.append(img_array)
training_images = np.array(training_images)

print(f"Training batch shape: {training_images.shape}")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, batch in enumerate(datagen_ex1.flow(training_images, batch_size=1)):
    axes[i].imshow(array_to_img(batch[0]))
    axes[i].set_title(f'🔄 Augmented {i+1}')
    axes[i].axis('off')
    if i >= 3:
        break
plt.suptitle('✅ Exercise 1 — Geometric Augmentation on Real Images', fontsize=13)
plt.tight_layout()
plt.show()

### Exercise 2 — Feature-wise & Sample-wise Normalisation on Real Images

In [ ]:
datagen_ex2 = ImageDataGenerator(
    featurewise_center=True,
    featurewise_std_normalization=True,
    samplewise_center=True,
    samplewise_std_normalization=True
)
datagen_ex2.fit(training_images)

print(f"Feature-wise mean: {datagen_ex2.mean}")
print(f"Feature-wise std:  {datagen_ex2.std}")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, batch in enumerate(datagen_ex2.flow(training_images, batch_size=1)):
    display_img = np.clip(array_to_img(batch[0]), 0, 255)
    axes[i].imshow(display_img)
    axes[i].set_title(f'📐 Normalised {i+1}')
    axes[i].axis('off')
    if i >= 3:
        break
plt.suptitle('✅ Exercise 2 — Feature-wise & Sample-wise Normalisation', fontsize=13)
plt.tight_layout()
plt.show()

### Exercise 3 — Custom Gaussian Noise on Real Images

In [ ]:
def add_random_noise(image):
    noise = np.random.normal(loc=0.0, scale=0.1, size=image.shape)
    return image + noise

datagen_ex3 = ImageDataGenerator(preprocessing_function=add_random_noise)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, batch in enumerate(datagen_ex3.flow(training_images, batch_size=1)):
    axes[i].imshow(np.clip(array_to_img(batch[0]), 0, 255))
    axes[i].set_title(f'⚡ Noisy {i+1}')
    axes[i].axis('off')
    if i >= 3:
        break
plt.suptitle('✅ Exercise 3 — Custom Augmentation: Gaussian Noise on Real Images', fontsize=13)
plt.tight_layout()
plt.show()

---

## 📊 Summary

| Technique | Key parameter | When to use | Formula |
|---|---|---|---|
| Geometric augmentation | `rotation_range`, `zoom_range`, ... | Small datasets, spatial invariance | $T(x,y) \rightarrow (x', y')$ |
| Feature-wise normalisation | `featurewise_center=True` + `.fit(X)` | Global dataset standardisation | $x' = (x - \mu_{\text{dataset}}) / \sigma_{\text{dataset}}$ |
| Sample-wise normalisation | `samplewise_center=True` | Per-image AGC | $x' = (x - \mu_{\text{sample}}) / \sigma_{\text{sample}}$ |
| Custom noise injection | `preprocessing_function=` | Sensor noise robustness | $x' = x + \mathcal{N}(0, \sigma^2)$ |

**Key takeaways:**
- `ImageDataGenerator` generates augmentations on-the-fly — no extra disk space
- `featurewise_*` requires `.fit()` first; `samplewise_*` does not
- `preprocessing_function` runs after all built-in transforms, before returning the batch
- Augmentation is **training-only** — validation generators should have no augmentation
- Augmentation as regularisation: $\mathcal{L}_{\text{eff}} = \mathbb{E}_{T \sim p(T)}[\mathcal{L}(f(T(x)), y)]$

---

## 🧪 Sandbox

In [ ]:
# SANDBOX 1: Combined augmentation — geometric + Gaussian noise
def add_gaussian_noise(image, sigma=0.05):
    noise = np.random.normal(0, sigma, image.shape)
    return np.clip(image + noise, 0, 255)

datagen_combined = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='reflect',
    preprocessing_function=add_gaussian_noise
)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, batch in enumerate(datagen_combined.flow(x, batch_size=1)):
    row, col = divmod(i, 4)
    axes[row][col].imshow(np.clip(batch[0], 0, 255).astype('uint8'))
    axes[row][col].set_title(f'Combined {i+1}')
    axes[row][col].axis('off')
    if i >= 7:
        break
plt.suptitle('🧪 Sandbox 1 — Geometric + Gaussian Noise Combined', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# SANDBOX 2: IQ-analogy noise — R=I, G=Q, B=carrier (telecom analogy)
def iq_noise_augmentation(image):
    sigma_iq = 15.0
    sigma_b  = 5.0
    image_noisy = image.copy()
    image_noisy[:, :, 0] += np.random.normal(0, sigma_iq, image[:, :, 0].shape)
    image_noisy[:, :, 1] += np.random.normal(0, sigma_iq, image[:, :, 1].shape)
    image_noisy[:, :, 2] += np.random.normal(0, sigma_b,  image[:, :, 2].shape)
    return image_noisy

datagen_iq = ImageDataGenerator(preprocessing_function=iq_noise_augmentation)
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, batch in enumerate(datagen_iq.flow(x, batch_size=1)):
    axes[i].imshow(np.clip(batch[0], 0, 255).astype('uint8'))
    axes[i].set_title(f'📡 IQ-Noise {i+1}')
    axes[i].axis('off')
    if i >= 3:
        break
plt.suptitle('🧪 Sandbox 2 — IQ Channel Noise (R=I, G=Q, B=carrier)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# SANDBOX 3: Original vs. Augmented CIFAR-10 images side by side
cifar_batch = x_train[:4] * 255.0
datagen_cifar = ImageDataGenerator(
    rotation_range=20, horizontal_flip=True,
    zoom_range=0.1, width_shift_range=0.1, fill_mode='nearest'
)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for j in range(4):
    axes[0][j].imshow(x_train[j])
    axes[0][j].set_title(f'Original: {CLASS_NAMES[y_train[j][0]]}')
    axes[0][j].axis('off')
for i, batch in enumerate(datagen_cifar.flow(cifar_batch, batch_size=4)):
    for j in range(4):
        axes[1][j].imshow(np.clip(batch[j], 0, 255).astype('uint8'))
        axes[1][j].set_title('Augmented')
        axes[1][j].axis('off')
    break
plt.suptitle('🧪 Sandbox 3 — Original vs. Augmented CIFAR-10', fontsize=13)
plt.tight_layout()
plt.show()